In [ ]:
import glob
import os
import shutil

REPO = "/kaggle/working/small-model-alignment"
found = glob.glob("/kaggle/input/**/pa2-code/**/alignment_core.py", recursive=True)
print(found)
if not os.path.exists(REPO):
    shutil.copytree(os.path.dirname(found[0]), REPO)
os.chdir(REPO)
os.environ["PYTHONPATH"] = REPO
print(os.listdir(REPO))

In [ ]:
!nvidia-smi
!python -m pip uninstall -y -q torchao || true
!python -m pip install -q "transformers>=4.43.0,<5" "datasets>=2.18,<5" huggingface_hub accelerate tqdm pyyaml sentencepiece safetensors pytest
!python -m pip install -q -e . --no-deps
assert _exit_code == 0

In [ ]:
import datasets
import torch
import transformers

print("torch", torch.__version__)
print("transformers", transformers.__version__)
print("datasets", datasets.__version__)
print("gpus", torch.cuda.device_count())
print("model proxy available", "MODEL_PROXY_URL" in os.environ)

In [ ]:
!python -m pip install -q "trl==0.29.1" "peft>=0.17,<0.22" "transformers>=4.56.2,<5" "datasets>=3.0,<5" "accelerate>=1.4.0"
assert _exit_code == 0

In [ ]:
!python scripts/train_lora_dpo.py --output_dir outputs/lora_dpo --epochs 3
assert _exit_code == 0

In [ ]:
!python scripts/plot_metrics.py --metrics_path outputs/lora_dpo/metrics.jsonl --output_dir outputs/plots --run_name lora_dpo --kind lora
assert _exit_code == 0

In [ ]:
!python scripts/generate.py --model_path HuggingFaceTB/SmolLM2-360M-Instruct --adapter_path outputs/lora_dpo/final --output_path outputs/generations/sanity_smol_lora_persona.json --generator smollm2-360m-lora-dpo --limit 3 --eval_path eval_sets/persona_style_eval.json --dataset persona_style_eval --prompt_format chat --max_new_tokens 256 --repetition_penalty 1.1
assert _exit_code == 0

In [ ]:
!python scripts/generate.py --model_path HuggingFaceTB/SmolLM2-360M-Instruct --output_path outputs/generations/smol_base_persona.json --generator smollm2-360m-instruct --eval_path eval_sets/persona_style_eval.json --dataset persona_style_eval --prompt_format chat --max_new_tokens 256 --repetition_penalty 1.1
assert _exit_code == 0

In [ ]:
!python scripts/generate.py --model_path HuggingFaceTB/SmolLM2-360M-Instruct --adapter_path outputs/lora_dpo/final --output_path outputs/generations/smol_lora_persona.json --generator smollm2-360m-lora-dpo --eval_path eval_sets/persona_style_eval.json --dataset persona_style_eval --prompt_format chat --max_new_tokens 256 --repetition_penalty 1.1
assert _exit_code == 0

In [ ]:
!python scripts/build_pairs.py --baseline_path outputs/generations/smol_base_persona.json --candidate_path outputs/generations/smol_lora_persona.json --output_path outputs/eval/lora_dpo_vs_base_pairs.jsonl
assert _exit_code == 0

In [ ]:
!zip -q -r /kaggle/working/lora_dpo_run_outputs.zip outputs -x 'outputs/sft/*' 'outputs/dpo_hh/*' 'outputs/dpo_ultrafeedback/*'
assert _exit_code == 0